<a href="https://colab.research.google.com/github/VictoriaO26/AI-assisted-Cloud-Infrastructure-Cybersecurity-Bootcamp/blob/main/Module_01_lab_Student_Notebook.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Module 1 - Cloud CI Network Security  

**Objective**: The student will learn to build, evaluate, and interpret a machine-learning-based network intrusion detection system (IDS), including how to handle severe class imbalance and how to diagnose the model's false positives and false negatives.

**Acknowledgement**: This module was developed under the National Science Foundation awards # 2416990, 2416992 and 2607393 at Tennessee Tech University, North Carolina A&T State University and University at Albany.

# AWS-Based Intrusion Detection System Architecture

The deployment of an Intrusion Detection System (IDS) in a cloud environment requires the orchestration of a traffic mirroring pipeline that captures and analyzes network activity in real-time. By utilizing a dedicated IDS node, raw traffic is diverted from production network interfaces to an isolated analysis environment without impacting performance. Once the mirrored traffic arrives at the IDS node, it is processed into flow-based statistical records, which serve as the input for machine learning inference. This approach allows for scalable, non-intrusive monitoring, where pre-trained models classify traffic patterns as either benign or malicious based on key network features.

---

### Implementation Workflow

1. **IDS Instance Deployment:** Provision an EC2 instance within a private subnet to serve as the dedicated analysis engine. This instance must be pre-configured with the required machine learning environment, pre-trained model artifacts, and the feature extraction pipeline.
2. **Traffic Mirroring Configuration:** Establish a **Traffic Mirror Target** associated with the IDS instance and configure a **Traffic Mirror Session** to duplicate ingress and egress traffic from target production instances to the IDS node.
3. **Data Capture and Flow Aggregation:** Utilize `tcpdump` on the IDS instance to intercept mirrored packets, feeding them into `NFStream` to aggregate raw data into 5-tuple statistical flows.
4. **Feature Extraction:** Process the flow data into the specific n-feature vector format required for model compatibility, ensuring normalization parameters match those used during initial training.
5. **Real-time Inference:** Pass the extracted feature vectors through the loaded model via `model.predict()` to generate classification labels and trigger automated responses or security alerts.

# Intrusion Detection on the HIKARI-2021 Dataset

This notebook will cover a network intrusion detection classifier on top of the
HIKARI-2021 flow-based dataset. It covers data cleaning, a leakage-safe
train/test split, class imbalance handling, model training and evaluation,
and model explainability with SHAP.


In [ ]:
# Install/upgrade the third-party libraries this notebook depends on.
# Several of these ship with Colab already, but pinning them here keeps
# the environment reproducible across sessions and machines.
!pip install -q --upgrade imbalanced-learn shap xgboost seaborn

In [ ]:
# ----------------------------------------------------------------------
# Standard library
# ----------------------------------------------------------------------
import os
import random
# ----------------------------------------------------------------------
# Data handling
# ----------------------------------------------------------------------
import numpy as np
import pandas as pd
# ----------------------------------------------------------------------
# Visualization
# ----------------------------------------------------------------------
import matplotlib.pyplot as plt
import seaborn as sn
from matplotlib.patches import Patch
# ----------------------------------------------------------------------
# Preprocessing, resampling, and model selection
# ----------------------------------------------------------------------
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import (train_test_split, StratifiedKFold, cross_val_score,)
from imblearn.over_sampling import SMOTE
from sklearn.utils.class_weight import compute_sample_weight
# ----------------------------------------------------------------------
# Modeling and evaluation
# ----------------------------------------------------------------------
import xgboost
from sklearn import metrics
from sklearn.metrics import (classification_report, precision_recall_curve, roc_curve,)
# ----------------------------------------------------------------------
# Model explainability
# ----------------------------------------------------------------------
import shap
# A fixed seed is used throughout the notebook so that splits, resampling,
# and random selections are reproducible from run to run.
RANDOM_SEED = 42
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

## 1. Dataset Acquisition

The dataset is pulled from Github via the LFS media point

In [ ]:
url = "https://media.githubusercontent.com/media/acr-research-lab/AI4SecureCI/main/module_1/ALLFLOWMETER_HIKARI2021.csv"
data = pd.read_csv(url)
data.head()

## 2. Loading and Cleaning the Data

Network flow datasets commonly contain missing values, and — because
several features are computed as rates (e.g. bytes per second) — divide-
by-zero cases that surface as infinite values. Both are removed before any
further processing so they don't distort scaling or model training later.

In [ ]:
# Replace infinite values (a common artifact of rate-based flow features)
# with NaN so they're caught by the same cleaning step as missing values.
## apply the replacement here
###################

###################

data.dropna(inplace=True)
data.reset_index(drop=True, inplace=True)

data.head()


## 3. Exploring the Class Distribution

Before encoding anything, it's worth looking at how traffic is distributed
across the fine-grained attack categories as well as the coarser
benign/attack label, since this shapes the resampling strategy used later.

In [ ]:
# use pd.dataframe[feature].value_count() to show the distribustion of the binary and multiclass labels.
###################

###################


## 4. Encoding the Traffic Category

The fine-grained traffic category is a categorical string; it's mapped to
an integer so it can be used as a model target and manipulated numerically
during resampling.

In [ ]:
# use 0-n for n multiclass labels and 0-1 for binary labels. The binary label is used for the first classification task and the
# multiclass label is used for the second classification task.

# Remember, for binary classification task, Benign and Background are the normal traffic and the rest are attack classes.

dic_map = {
    'Benign': 0,
    'Background': 1,
    'Probing': 2,
    'Bruteforce': 3,
    'Bruteforce-XML': 4,
    'XMRIGCC CryptoMiner': 5,
}

## apply the dictionary map on traffic_category column

###################

###################


## 5. Dropping Identifier Columns

The first several columns are per-flow identifiers (flow IDs, source/
destination IPs and ports, timestamps). These don't generalize — a model
that learns from raw IP addresses would just memorize which hosts were
used to generate attacks in this particular capture, rather than learning
transferable traffic patterns. They're dropped before modeling.

In [ ]:
# Identify the irrelevant identifier columns and remove them from your structured data
data = data[list(data.columns[<relevant first column index>:])]


## 6. Train/Test Split (Before Any Scaling)

This ordering matters: the data is split into train and test partitions
**before** any feature scaling or resampling happens. If scaling were fit
on the full dataset first, statistics from the test set — which is meant
to represent unseen data — would leak into preprocessing, producing an
overly optimistic view of how well the model generalizes. Splitting first
and fitting every downstream preprocessing step only on the training
partition avoids that leakage.

In [ ]:
# Please use 80:20 ratio train:test. We can check on other splitting ratio like 70:30, 60:40, 50:50 etc. but for now, we will use 80:20 ratio.

train_data, test_data = train_test_split(
    data, random_state=RANDOM_SEED, test_size=<test_ratio>, shuffle=True
)


## 7. Feature Scaling

The scaler is fit only on the training partition's numeric features, then
applied to both partitions. This keeps the test set genuinely unseen
during preprocessing.

In [ ]:
# Please use StandardScaler to scale the features.

label_cols = ['traffic_category', 'Label']

# define the scaler, fit it with only the feature data form train_data
###################

###################

def scale_partition(partition):
    """Scale a partition's numeric features using the training-fitted
    scaler, leaving the label columns untouched and the row order intact."""
    numeric = partition.drop(columns=label_cols)
    labels = partition[label_cols].reset_index(drop=True)
    scaled = pd.DataFrame(scaler.transform(numeric), columns=numeric.columns)
    return pd.concat([scaled, labels], axis=1)

train_data = scale_partition(train_data)
test_data = scale_partition(test_data)

train_data.head()


## 8. Oversample the Rare Attack Categories with SMOTE

The raw traffic categories are heavily skewed toward benign/background
traffic, with some attack types represented by only a handful of flows.
Two complementary steps are used to address this:

1. Rare attack categories are moderately oversampled with SMOTE, so each
   one has a reasonable number of real-and-synthetic examples to learn
   from.
2. The dominant benign class is then undersampled down to a fixed ratio
   relative to the attack class — n benign examples for every 1 attack
   example — rather than forcing a full 1:1 balance. Forcing an exact
   1:1 balance removes so much real benign data that the model ends up
   with a distorted view of what "normal" traffic looks like, which in
   practice hurts precision and recall on the attack class. A 2:1 ratio as we already multiplied the attack instances using SMOTE and thus
   keeps far more of the real benign signal while still giving attacks
   much more relative weight than their natural frequency.

In [ ]:
# Apply your SMOTE resampling on the training data to balance the classes. Don't touch the test data. Use random_state=RANDOM_SEED for
# reproducibility. You can use your own ration. Let us use as below:

# You can skip the Benign and Background classes cause they are the majority classes.
# For the other minority classes, we can use the following:
    # take a random ratio lets call it x ranging 0.15-0.25
    # and generate samples for that specific class using SMOTE as x*total sample count for majority Benign class.
# Or you can use just a single constant factor for all the minority classes. For example, if the majority class has
# 1000 samples and you want to generate 20% of that for each minority class, then you will generate 200 samples for each minority class.

target_col = 'traffic_category'
binary_label_col = 'Label'

X_train = train_data.drop(columns=[target_col, binary_label_col])
y_train = train_data[target_col]

class_counts = y_train.value_counts()
majority_class = class_counts.idxmax()
majority_count = class_counts.max()

# A dedicated, seeded random generator drives the oversampling ratios so
# the resulting class sizes are reproducible from run to run, independent
# of any other randomness elsewhere in the notebook.
ratio_rng = np.random.default_rng(RANDOM_SEED)

# Build a sampling target for any minority class that falls under a
# 15-25% share of the majority class's size.
sampling_strategy = {}
for class_label, count in class_counts.items():
  # randomized ratio for better generalized approach
    desired_minority_count = int(majority_count * ratio_rng.uniform(<lower_range>, <upper_range>)) # provide the range here
    if class_label != majority_class and count < desired_minority_count:
        sampling_strategy[class_label] = desired_minority_count

sm = SMOTE(sampling_strategy=sampling_strategy, random_state=RANDOM_SEED)
X_resampled, y_resampled = sm.fit_resample(X_train, y_train)


# 9. Rebuild the Balanced Training DataFrame

In [ ]:
# Reassemble the resampled features into a DataFrame and re-attach the multi-class target as a new column
# recheck the value count for the target labels

train_data_balanced = pd.DataFrame(X_resampled, columns=<_all the feature columns>)

# Re-attach the multi-class target. Appending as a plain column assignment
# (rather than inserting at a hardcoded position) keeps this robust to any
# future change in the number of feature columns.
train_data_balanced[target_col] = y_resampled

# Re-derive the binary label from the (possibly synthetic) traffic category:
# categories 0 (Benign) and 1 (Background) are non-malicious traffic,
# everything else is an attack.
train_data_balanced[binary_label_col] = train_data_balanced[target_col].apply(
    lambda x: 0 if x in (0, 1) else 1
)

train_data = train_data_balanced
train_data.head()


# 10. Undersampling the Normal Class to a 2:1 Ratio

With the rare attack categories now topped up by SMOTE, the benign side of
the binary label is undersampled so it's no longer overwhelmingly
dominant — capped at 2 times the size of the attack class rather than
matched 1:1 against it.

In [ ]:
# Do your random undersampling on the benign case only to make it more generalized. Here you will make the normal traffic class (Benign)
# to be approximately 2 times to the sum of all the attack classes.

attack_rows = train_data[train_data.Label == 1]
benign_rows = train_data[train_data.Label == 0]

n_times =   # provide the exact ratio value

benign_target_count = min(len(benign_rows), len(attack_rows) * n_times)
benign_rows = benign_rows.sample(n=benign_target_count, random_state=RANDOM_SEED)

train_data = pd.concat([attack_rows, benign_rows], ignore_index=False)
train_data = train_data.sample(frac=1, random_state=RANDOM_SEED)  # shuffle

train_data.Label.value_counts()



# 11. Verify the final Resulting Distribution

In [ ]:
# Again please recheck the final target labels value count so check its 2:1 ratio for Normal:Attack classes.

###################

###################



# 12. Preparing Feature/Label Arrays

Features and labels are extracted for both partitions and converted to
NumPy arrays for use with XGBoost. Because the two label columns are
always the last two columns after the steps above, they can be sliced off
positionally.

In [ ]:
# Separate the feature values and target label values in order.

###################
# X_train, X_test = # select appropriate X portion
# y_train, y_test = # select appropriate Y portion
###################



feature_names = train_data.columns[:-2]

X_train, X_test = np.array(X_train), np.array(X_test)
y_train, y_test = np.array(y_train), np.array(y_test)

unique_classes = ['Benign', 'Attack']


# 13. Class Weighting for the Binary Classifier

Even after the SMOTE and 2:1 undersampling steps above, the binary label
is still imbalanced by design (2 benign examples for every attack
example). `scale_pos_weight` gives XGBoost an extra nudge toward the
minority class on top of that, computed from the training partition only
so the test set has no influence on it.

In [ ]:
###################
# scale_pos_weight = count(negative) / count(positive) or count(class=0) / count(class=1)
###################


# 14. Training the Binary (Benign vs. Attack) Classifier

In [ ]:
# train a xgboost classifier on the prepared training data. Use appropriate objective parameter for loss function.


xgb_classifier = xgboost.XGBClassifier(
    objective=<objective>,
    n_estimators=100,
    max_depth=5,
    scale_pos_weight=scale_pos_weight,
    random_state=RANDOM_SEED,
    verbosity=0,
)

xgb_model = xgb_classifier.fit(X_train, y_train)
xgb_predicted_values = xgb_model.predict(X_test)



# 15. Evaluating the Binary Classifier

In [ ]:
# Generate the classification report

classification_report(<placeholder original_ y val>, <placeholder predicted y val>, target_names=unique_classes)



In [ ]:
# Generate the metrics for the model acc, precision, recall, f1-score

print(metrics.accuracy_score(<placeholder original_ y val>, <placeholder predicted y val>))
print(metrics.precision_score(<placeholder original_ y val>, <placeholder predicted y val>, average='weighted'))
print(metrics.recall_score(<placeholder original_ y val>, <placeholder predicted y val>, average='weighted'))
print(metrics.f1_score(<placeholder original_ y val>, <placeholder predicted y val>, average='weighted'))

# 16. Cross-Validation on the Training Partition

To sanity-check that the reported test performance isn't a fluke of one
particular split, stratified cross-validation is run — but only across the
training partition, never touching the held-out test set. Running
cross-validation over the full dataset (including rows that were held out
for the final test) would let the test set leak back into the evaluation.

In [ ]:
num_folds = 5


skf = StratifiedKFold(n_splits=num_folds, shuffle=True, random_state=RANDOM_SEED)

st_cross_val_results = cross_val_score(xgb_classifier, <placeholder train x val>, <placeholder train y val>, cv=skf)
print(f'Stratified Cross-Validation Results (Accuracy): {st_cross_val_results}')
print(f'Mean Accuracy: {st_cross_val_results.mean()}')


# 17. Build a Confusion Matrix

In [ ]:
# Plot a confusion matrix


xgb_cm = metrics.confusion_matrix(<placeholder original_ y val>, <placeholder predicted y val>)

fig, ax = plt.subplots()
sn.heatmap(xgb_cm, annot=True, fmt='d', ax=ax, cmap=plt.cm.Blues, cbar=False)
ax.set(xlabel="Pred", ylabel="True", xticklabels=unique_classes, yticklabels=unique_classes)
plt.rcParams["figure.figsize"] = (10, 6)
plt.yticks(rotation=0)
plt.rcParams.update({'font.size': 22, 'font.weight': 'bold'})
plt.show()


# 18. False Positive / False Negative Rates

For an intrusion detection system, the false positive rate (benign traffic
flagged as an attack) and false negative rate (attacks that slip through)
are often more operationally meaningful than accuracy alone.

In [ ]:
# calculate the FPR and FNR

def calculate_fpr_fnr(confusion_matrix):
    """
    Calculate and print the False Positive Rate (FPR) and False Negative
    Rate (FNR) from a 2x2 confusion matrix.

    Parameters:
    - confusion_matrix: 2D array-like, the confusion matrix

    Returns:
    - FPR: False Positive Rate
    - FNR: False Negative Rate
    """
    TN, FP, FN, TP = confusion_matrix.ravel()

    FPR = FP / (FP + TN)
    FNR = FN / (FN + TP)

    print(f"False Positive Rate (FPR): {FPR:.4f}")
    print(f"False Negative Rate (FNR): {FNR:.4f}")

    return FPR, FNR


# Call the function and put the y_true and y_pred val
###################

###################


# 19. Probability Calibration (Brier Score)

In [ ]:
# Predicted probabilities for the positive (Attack) class and calculate the brier score.


# Predicted probabilities for the positive (Attack) class.
y_prob = <placeholder probability of label attack>

brier_score_pos = metrics.brier_score_loss(<placeholder original y>, y_prob)
print(f'Brier score (binary): {brier_score_pos:.4f}')


# 20. Precision-Recall and ROC Curves

In [ ]:
# generate two plots for the given task and observe the tradeoff between precision and recall and the ROC curve. Use the predicted
# probabilities for the positive (Attack) class.

precision, recall, thresholds = precision_recall_curve(<placeholder original y>, y_prob)
plt.plot(recall, precision, marker='.')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.show()

fpr, tpr, _ = roc_curve(<placeholder original y>, y_prob)
plt.plot(fpr, tpr, marker='.')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.show()



# 21. Segmenting Predictions into TP / TN / FP / FN

These four groups of predictions are separated out so that model behavior
can be inspected group by group in the explainability section below.

In [ ]:
# fill up the others using the same code but with different logics for TN, FP, and FN
###################
# tp_instances = np.where((y_test_arr == 1) & (pred_arr == 1))[0]
# tn_instances =
# fp_instances =
# fn_instances =
###################

## 22. Confidence Threshold Sensitivity

Rather than only looking at the default 0.5 decision threshold, it's worth
checking how confident the model is across a range of thresholds within
each prediction group — this shows whether errors tend to be made with
high or low confidence.

In [ ]:
# Observe the soft label probabilities for the all four groups (TP, TN, FP, FN) and seee their distribution in different threshold like 0.7-0.9

for threshold in (<provide multiple ranges values>):
    complement = 1 - threshold
    print(f'--- Threshold: {threshold:.2f} ---')
    print('TP with prob >= threshold:  ', len([p for p in y_prob[tp_instances] if p >= threshold]) * 100 / len(tp_instances))
    print('TN with prob <= complement: ', len([p for p in y_prob[tn_instances] if p <= complement]) * 100 / len(tn_instances))
    print('FP with prob >= threshold:  ', len([p for p in y_prob[fp_instances] if p >= threshold]) * 100 / len(fp_instances))
    print('FN with prob <= complement: ', len([p for p in y_prob[fn_instances] if p <= complement]) * 100 / len(fn_instances))
    print()

# 23. Model Explainability with SHAP - Overlapping plot Function

SHAP values are used to understand which features drive individual
predictions, and how that differs across correctly and incorrectly
classified traffic. Two helper functions are defined first: one to plot
two overlapping sets of SHAP values for comparison, and one to pull out
the top contributing features for a group of predictions.

In [ ]:
def plot_overlapping_shap(shap_values_instance1, shap_values_instance2, feature_names, feature_values, label_exp, raw_prediction_prob=None, pred_val=None, color_instance1=None, color_instance2=None, bar_width1=0.75, bar_width2=0.9):
    """
    Plot two overlapping SHAP bar plots for two instances (or an instance
    against a group average) so their feature contributions can be compared
    directly.

    Parameters:
    - shap_values_instance1: 1D array-like, SHAP values for the first instance.
    - shap_values_instance2: 1D array-like, SHAP values for the second instance.
    - feature_names: list of feature names.
    - feature_values: list of feature values.
    - label_exp: label for the second instance.
    - raw_prediction_prob: Probability of the prediction to be displayed as text.
    - pred_val: Prediction value to be displayed as text.
    - color_instance1: tuple or list of colors for the first instance, default to bright colors.
    - color_instance2: tuple or list of colors for the second instance, default to transparent colors.
    - bar_width1: width of the bars for the first instance.
    - bar_width2: width of the bars for the second instance.
    """

    light_red, light_green = (1.0, 0.71, 0.71, 0.5), (0.56, 0.93, 0.56, 0.5)

    if color_instance1 is None:
        color_instance1 = ['deepskyblue' if v < 0 else 'deeppink' for v in shap_values_instance1]
    if color_instance2 is None:
        color_instance2 = [light_green if v < 0 else light_red for v in shap_values_instance2]

    shap_values_instance1 = np.array(shap_values_instance1)
    shap_values_instance2 = np.array(shap_values_instance2)

    assert (len(shap_values_instance1) == len(shap_values_instance2)
            == len(feature_names) == len(feature_values)), "All inputs must have the same length"

    feature_labels = [f"{name} = {round(value, 4)}" for name, value in zip(feature_names, feature_values)]

    fig, ax = plt.subplots()

    bar1 = ax.barh(np.arange(len(feature_names)), shap_values_instance1, height=bar_width1, color=color_instance1, label='Instance 1')
    bar2 = ax.barh(np.arange(len(feature_names)), shap_values_instance2, height=bar_width2, color=color_instance2, label=label_exp)

    ax.set_yticks(np.arange(len(feature_names)))
    ax.set_yticklabels(feature_labels)
    ax.set_xlabel('SHAP value')

    title_text = ""
    if pred_val is not None:
        if int(pred_val) == 0:
            if raw_prediction_prob is not None:
                title_text += f"Raw Prediction Probability: {1 - raw_prediction_prob:.3f}\n"
                title_text += f"Prediction Value: {pred_val:.0f}"
        else:
            if raw_prediction_prob is not None:
                title_text += f"Raw Prediction Probability: {raw_prediction_prob:.3f}\n"
                title_text += f"Prediction Value: {pred_val:.0f}"
    plt.title(title_text.strip())

    legend_elements = [
        Patch(facecolor='deeppink', edgecolor='deeppink', label='Instance 1 (positive)'),
        Patch(facecolor='deepskyblue', edgecolor='deepskyblue', label='Instance 1 (negative)'),
        Patch(facecolor=light_red, edgecolor=light_red, label=f'{label_exp} (positive)'),
        Patch(facecolor=light_green, edgecolor=light_green, label=f'{label_exp} (negative)')
    ]
    ax.legend(handles=legend_elements)

A quick sanity check of the plotting helper against small synthetic values,
just to confirm it renders correctly before it's used on real SHAP output.

In [ ]:
shap_values_instance1 = [0.1, 0.2, -0.1, 0.3, -0.2]
shap_values_instance2 = [0.15, 0.25, -0.05, 0.25, -0.2]
f_names = ['Feature 1', 'Feature 2', 'Feature 3', 'Feature 4', 'Feature 5']
feature_values = [1.1, 2.2, 3.3, 4.4, 5.5]
label_exp = 'Specific Instance'

plot_overlapping_shap(shap_values_instance1, shap_values_instance2, f_names, feature_values, label_exp, raw_prediction_prob=0.0, pred_val=0)

In [ ]:
def top_20_features(shap_value_group):
    """Return the indices of the 20 features with the largest mean
    absolute SHAP value for a group of predictions, ordered ascending."""
    abs_shap_values = np.abs(shap_value_group)
    top_20_indices = np.argsort(abs_shap_values)[-20:]
    sorted_top_20_indices = top_20_indices[np.argsort(abs_shap_values[top_20_indices])]
    return sorted_top_20_indices

# 24. Build an Explainer

In [ ]:
# an example is given to you for calculating SHAP values for an xgboost model and for the TP instances.
# you need to pass the true positives instances to the explainer to generate the SHAP values for the true positives instances.

# explainer = shap.TreeExplainer(xgb_classifier, feature_perturbation="tree_path_dependent")
# tp_shap_values = explainer(X_test[tp_instances], check_additivity=False)

In [ ]:
# tp_shap_values.shape

In [ ]:
# set your feature names from the saved dataframe
# feature_names =

In [ ]:
# tp_shap_values.feature_names = feature_names

# 25. True Positives: which features drive correctly caught attacks

In [ ]:
# shap.plots.bar(tp_shap_values.mean(0), max_display=20) #bar plot with actual mean values
# shap.summary_plot(tp_shap_values, plot_type="violin", class_names=unique_classes, plot_size=0.2, max_display=20) #beeswarm plot with actual feature value context

In [ ]:
# This will be needed to plot overlapping plots for a randomly picked true positive instances SHAP values and the mean SHAP values for all true positive instances.

# tp_shap_values_mean = tp_shap_values.values.mean(0)
# tp_top_20_features = top_20_features(tp_shap_values_mean)
# tp_top_20_features

In [ ]:
# this will plot the overlapping SHAP values for a randomly picked true positive instance and the mean SHAP values for all true positive instances.

# rand_idx = random.choice(tp_instances)
# print(rand_idx)
# tp_random_case = explainer([X_test[rand_idx]], check_additivity=False)
# plot_overlapping_shap(tp_random_case.values[0][tp_top_20_features], tp_shap_values_mean[tp_top_20_features], feature_names[tp_top_20_features], tp_random_case.data[0][tp_top_20_features], 'TP_Mean', raw_prediction_prob=y_prob[rand_idx], pred_val=xgb_predicted_values[rand_idx])
# plt.rcParams["figure.figsize"] = (6, 7)
# plt.yticks(rotation=0)
# plt.rcParams.update({'font.size': 10, 'font.weight': 'bold'})
# plt.show()

### True Negatives: which features drive correctly identified benign traffic

In [ ]:
# Apply the same for the true negative instances as well following the above TP example. Just pass the TN instances to the explainer.

# 26. False Positives: which features mislead the model into flagging benign traffic as an attack

In [ ]:
# Apply the same for the false positive instances as well.

### False Negatives: which features mislead the model into missing real attacks

In [ ]:
# Apply the same for the false negative instances as well.

# 27. Spot-Checking Individual Predictions Against Group Averages

For each of the four prediction groups, a handful of random individual
cases are compared against that group's average SHAP profile. This helps
surface whether a group's behavior is driven consistently by the same
features, or whether individual cases vary a lot from the average.

In [ ]:
# an example for the TP case is given. It chooses 10 random instances from tp_instances and check visually that are those more overllaping with TP global mean bar chart or FP mean bar chart. More overlapping bars means more similarity.


# for i in range(10):
#     print('-' * 150)
#     rand_idx = random.choice(tp_instances)
#     tp_random_case = explainer([X_test[rand_idx]], check_additivity=False)
#     plot_overlapping_shap(tp_random_case.values[0][tp_top_20_features], tp_shap_values_mean[tp_top_20_features], feature_names[tp_top_20_features], tp_random_case.data[0][tp_top_20_features], 'TP_Mean', raw_prediction_prob=y_prob[rand_idx], pred_val=xgb_predicted_values[rand_idx])
#     plot_overlapping_shap(tp_random_case.values[0][fp_top_20_features], fp_shap_values_mean[fp_top_20_features], feature_names[fp_top_20_features], fp_random_case.data[0][fp_top_20_features], 'FP_Mean', raw_prediction_prob=y_prob[rand_idx], pred_val=xgb_predicted_values[rand_idx])
#     plt.rcParams["figure.figsize"] = (8, 6)
#     plt.yticks(rotation=0)
#     plt.rcParams.update({'font.size': 10, 'font.weight': 'bold'})
#     plt.show()

# 28. Multi-Class Classification (Attack Type Identification)

Beyond flagging benign vs. attack, it's often useful to identify *which
type* of attack traffic belongs to. A separate multi-class model is
trained on the same SMOTE-balanced training partition, using the
fine-grained traffic category as the target. The binary label is excluded
from the features so the model can't shortcut using it.

In [ ]:
# Drop the binary Label column so the model can't use it as a shortcut.
X_train_mult = train_data.drop(columns=['Label', 'traffic_category'])
y_train_mult = train_data['traffic_category']

# Apply the exact same column drops to the untouched testing partition.
X_test_mult = test_data.drop(columns=['Label', 'traffic_category'])
y_test_mult = test_data['traffic_category']

# Compute a weight for each training sample, inversely proportional to its class frequency
sample_weights = compute_sample_weight(class_weight='balanced', y=y_train_mult)**0.5

# objective='multi:softprob' lets the model predict class probabilities across all six categories.
model = xgboost.XGBClassifier(
    objective=<appropriate objective>,
    num_class=6,
    random_state=RANDOM_SEED,
    n_estimators=100,
    verbosity=0,
)

model.fit(X_train_mult, y_train_mult, sample_weight=sample_weights)

y_pred = model.predict(X_test_mult)

class_names = ['Benign', 'Background', 'Probing', 'Bruteforce', 'Bruteforce-XML', 'XMRIGCC CryptoMiner']

print("--- Multi-Class Classification Report ---")
print(classification_report(y_test_mult, y_pred, target_names=class_names))